# Benchmarking + Recommendation with MLflow

This notebook runs a SageMaker **AIBenchmarkJob** and **AIRecommendationJob**, logging results to a shared MLflow experiment so you can compare them side-by-side in the MLflow App UI.

**What you'll do:**
1. Install dependencies and configure your environment
2. Set up an IAM execution role with the required policies
3. Create an S3 output bucket
4. Run a benchmark against your deployed endpoint
5. Upload model weights to S3
6. Run a recommendation job to find optimal deployment configs
7. View results in MLflow and clean up

**Prerequisites:**
- An **MLflow App** in your account — create one via **SageMaker Studio → MLflow → Create MLflow app** ([docs](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow-app-setup.html))
- A **SageMaker endpoint** running an OpenAI-compatible model (already `InService`)

## Step 1: Install Dependencies & Configure

Update the configuration variables below for your environment (region, account, endpoint name, MLflow app ARN).

In [ ]:
!pip install -q -U boto3 botocore

In [ ]:
import json
import subprocess
import sys
import time
import uuid

import boto3
import botocore.config

# --- Configuration (update these for your environment) ---
REGION          = "us-east-1"
MLFLOW_APP_NAME = "<your-mlflow-app-name>"       # e.g. "app-XXXXXXXXXXXX" from SageMaker Studio
ENDPOINT_NAME   = "<your-endpoint-name>"         # e.g. "jumpstart-dft-hf-llm-qwen2-0-5b-..."

# Resolve account ID dynamically
session = boto3.Session(region_name=REGION)
sts = session.client("sts")
ACCOUNT_ID = sts.get_caller_identity()["Account"]

MLFLOW_APP_ARN = f"arn:aws:sagemaker:{REGION}:{ACCOUNT_ID}:mlflow-app/{MLFLOW_APP_NAME}"

# S3 bucket for job outputs — must contain 'sagemaker' in the name so
# AmazonSageMakerFullAccess allows s3:PutObject on it.
S3_OUTPUT_BUCKET = f"mlflow-sagemaker-{REGION}-{ACCOUNT_ID}"

sm = session.client(
    "sagemaker",
    config=botocore.config.Config(retries={"mode": "standard", "max_attempts": 15}),
)

print(f"Region:   {REGION}")
print(f"Account:  {ACCOUNT_ID}")
print(f"Endpoint: {ENDPOINT_NAME}")
print(f"boto3:    {boto3.__version__}")

## Step 2: IAM Role Setup

The execution role needs `AmazonSageMakerFullAccess` plus inline policies for MLflow data-plane access, service quotas, and endpoint invocation. If the role doesn't exist, the cell prints the CLI commands to create it.

In [ ]:
ROLE_NAME = "mlflow-tutorial-exec-role"

iam = session.client("iam")
try:
    role = iam.get_role(RoleName=ROLE_NAME)["Role"]
    ROLE_ARN = role["Arn"]
    print(f"✅ Role found: {ROLE_ARN}")
except iam.exceptions.NoSuchEntityException:
    print("❌ Role not found. Run these CLI commands as an admin to create it:")
    print(f"""
aws iam create-role --role-name {ROLE_NAME} --assume-role-policy-document '{{"Version":"2012-10-17","Statement":[{{"Effect":"Allow","Principal":{{"AWS":"arn:aws:iam::{ACCOUNT_ID}:root","Service":"sagemaker.amazonaws.com"}},"Action":"sts:AssumeRole"}}]}}'

aws iam attach-role-policy --role-name {ROLE_NAME} --policy-arn arn:aws:iam::aws:policy/AmazonSageMakerFullAccess

aws iam put-role-policy --role-name {ROLE_NAME} --policy-name MlflowDataPlaneAccess --policy-document '{{"Version":"2012-10-17","Statement":[{{"Effect":"Allow","Action":["sagemaker-mlflow:*","sagemaker:CallMlflowAppApi","sagemaker:DescribeMlflowApp"],"Resource":"*"}}]}}'

aws iam put-role-policy --role-name {ROLE_NAME} --policy-name ServiceQuotasAccess --policy-document '{{"Version":"2012-10-17","Statement":[{{"Effect":"Allow","Action":"servicequotas:GetServiceQuota","Resource":"*"}}]}}'

aws iam put-role-policy --role-name {ROLE_NAME} --policy-name EndpointInvokeAccess --policy-document '{{"Version":"2012-10-17","Statement":[{{"Effect":"Allow","Action":["sagemaker:InvokeEndpoint","sagemaker:InvokeEndpointWithResponseStream"],"Resource":"arn:aws:sagemaker:{REGION}:{ACCOUNT_ID}:endpoint/{ENDPOINT_NAME}"}}]}}'
""")

## Step 3: Create S3 Output Bucket

In [ ]:
s3_client = session.client("s3", region_name=REGION)

try:
    s3_client.head_bucket(Bucket=S3_OUTPUT_BUCKET)
    print(f"✅ Bucket already exists: {S3_OUTPUT_BUCKET}")
except Exception as e:
    error_code = getattr(e, "response", {}).get("Error", {}).get("Code", "")
    if error_code in ("404", "403", "NoSuchBucket"):
        s3_client.create_bucket(Bucket=S3_OUTPUT_BUCKET)
        print(f"✅ Created bucket: {S3_OUTPUT_BUCKET}")
    else:
        raise

## Step 4: MLflow Experiment Setup

Both jobs log to the same MLflow experiment for side-by-side comparison.

In [ ]:
MLFLOW_EXPERIMENT = "qwen2-0.5b-experiments"
SESSION_UID       = uuid.uuid4().hex[:6]
BENCH_RUN_NAME    = f"benchmark-{SESSION_UID}"
print(f"Experiment: {MLFLOW_EXPERIMENT}")
print(f"  benchmark run: {BENCH_RUN_NAME}")

## Step 5: Submit Benchmark Job

Runs a synthetic load test against your endpoint. The tokenizer must match the deployed model.

In [ ]:
TOKENIZER = "Qwen/Qwen2-0.5B-Instruct"

# Verify endpoint is reachable
assert "<" not in ENDPOINT_NAME, "Set ENDPOINT_NAME at the top of the notebook before running this cell."
ep_status = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointStatus"]
assert ep_status == "InService", f"Endpoint {ENDPOINT_NAME} status is {ep_status} — must be InService."

bench_workload = {
    "benchmark":  {"type": "aiperf"},
    "parameters": {
        "tokenizer": TOKENIZER,
        "concurrency": 1, "request_count": 3, "streaming": True,
        "prompt_input_tokens_mean": 32, "prompt_input_tokens_stddev": 8,
        "output_tokens_mean": 16, "output_tokens_stddev": 4,
        "random_seed": 42,
    },
    "tooling": {"api_standard": "openai", "version": "0.8.0"},
}

BENCH_CONFIG = f"mlflow-bench-cfg-{SESSION_UID}"
BENCH_JOB    = f"mlflow-bench-job-{SESSION_UID}"

sm.create_ai_workload_config(
    AIWorkloadConfigName=BENCH_CONFIG,
    AIWorkloadConfigs={"WorkloadSpec": {"Inline": json.dumps(bench_workload)}},
)
sm.create_ai_benchmark_job(
    AIBenchmarkJobName=BENCH_JOB,
    AIWorkloadConfigIdentifier=BENCH_CONFIG,
    RoleArn=ROLE_ARN,
    BenchmarkTarget={"Endpoint": {"Identifier": ENDPOINT_NAME}},
    OutputConfig={
        "S3OutputLocation": f"s3://{S3_OUTPUT_BUCKET}/mlflow-tutorial/{BENCH_JOB}/",
        "MlflowConfig": {
            "MlflowResourceArn":    MLFLOW_APP_ARN,
            "MlflowExperimentName": MLFLOW_EXPERIMENT,
            "MlflowRunName":        BENCH_RUN_NAME,
        },
    },
)
print(f"Job: {BENCH_JOB}")

### Wait for Benchmark (~10–15 min)

In [ ]:
TERMINAL = {"Completed", "Failed", "Stopped"}
deadline = time.time() + 30 * 60
last = ""
while time.time() < deadline:
    desc = sm.describe_ai_benchmark_job(AIBenchmarkJobName=BENCH_JOB)
    status = desc["AIBenchmarkJobStatus"]
    if status != last:
        print(f"  [{time.strftime('%H:%M:%S')}] {status}")
        last = status
    if status in TERMINAL:
        break
    time.sleep(30)
if status == "Failed":
    print(f"  FailureReason: {desc.get('FailureReason')}")

## Step 6: Upload Model Weights to S3

The recommendation job requires model weights in S3 to determine optimal instance types and configurations. This cell downloads **Qwen2-0.5B-Instruct** from Hugging Face Hub and streams it directly to your S3 bucket.

> **Already have your model in S3?** Skip this cell and set `MODEL_S3_URI` directly in Step 7 to your existing S3 path (e.g. `s3://my-bucket/models/my-model/`).

In [ ]:
import subprocess
import sys

# Install dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)

from huggingface_hub import list_repo_files
import urllib.request

MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"
S3_PREFIX = "models/qwen2-0.5b"
BASE_URL  = f"https://huggingface.co/{MODEL_ID}/resolve/main"

s3_client = session.client("s3", region_name=REGION)

# List all files in the repo
files = list(list_repo_files(MODEL_ID))
print(f"Found {len(files)} files to upload directly to S3...")

for filename in files:
    url = f"{BASE_URL}/{filename}"
    s3_key = f"{S3_PREFIX}/{filename}"
    print(f"  streaming: {filename} → s3://{S3_OUTPUT_BUCKET}/{s3_key}")
    
    # Stream from HuggingFace URL directly into S3 upload
    with urllib.request.urlopen(url) as response:
        s3_client.upload_fileobj(response, S3_OUTPUT_BUCKET, s3_key)

# Verify config.json landed
response = s3_client.list_objects_v2(Bucket=S3_OUTPUT_BUCKET, Prefix=S3_PREFIX)
keys = [obj["Key"] for obj in response.get("Contents", [])]
if any("config.json" in k for k in keys):
    MODEL_S3_URI = f"s3://{S3_OUTPUT_BUCKET}/{S3_PREFIX}/"
    print(f"\n✅ Upload complete. MODEL_S3_URI = {MODEL_S3_URI}")
else:
    print("\n❌ config.json not found — something went wrong.")


## Step 7: Submit Recommendation Job

The recommendation job provisions its own endpoints internally and finds optimal deployment configurations for your model.

In [ ]:
REC_SESSION_UID = uuid.uuid4().hex[:6]
REC_CONFIG = f"mlflow-rec-cfg-{REC_SESSION_UID}"
REC_JOB    = f"mlflow-rec-job-{REC_SESSION_UID}"
REC_RUN_NAME = f"recommendation-{REC_SESSION_UID}"
print(f"Rec Session UID: {REC_SESSION_UID}")

MODEL_S3_URI = f"s3://{S3_OUTPUT_BUCKET}/models/qwen2-0.5b/"
assert MODEL_S3_URI.startswith("s3://"), f"Invalid MODEL_S3_URI: {MODEL_S3_URI}"
print(f"Model S3 URI: {MODEL_S3_URI}")

rec_workload = {
    "benchmark":  {"type": "aiperf"},
    "parameters": {
        "prompt_input_tokens_mean": 1600, "prompt_input_tokens_stddev": 200,
        "output_tokens_mean": 600, "output_tokens_stddev": 100,
        "request_count": 100,
    },
    "tooling": {"api_standard": "openai", "version": "0.8.0"},
}

sm.create_ai_workload_config(
    AIWorkloadConfigName=REC_CONFIG,
    AIWorkloadConfigs={"WorkloadSpec": {"Inline": json.dumps(rec_workload)}},
)
sm.create_ai_recommendation_job(
    AIRecommendationJobName=REC_JOB,
    ModelSource={"S3": {"S3Uri": MODEL_S3_URI}},
    OutputConfig={
        "S3OutputLocation": f"s3://{S3_OUTPUT_BUCKET}/mlflow-tutorial/{REC_JOB}/",
        "MlflowConfig": {
            "MlflowResourceArn":    MLFLOW_APP_ARN,
            "MlflowExperimentName": MLFLOW_EXPERIMENT,
            "MlflowRunName":        REC_RUN_NAME,
        },
    },
    ComputeSpec={
        "InstanceTypes": ["ml.g6.12xlarge"]
    },
    RoleArn=ROLE_ARN,
    AIWorkloadConfigIdentifier=REC_CONFIG,
    PerformanceTarget={"Constraints": [{"Metric": "ttft-ms"}]},
    OptimizeModel=False,
)
print(f"Job: {REC_JOB}")

### Wait for Recommendation (~30 min – 2 hours)

In [ ]:
TERMINAL = {"Completed", "Failed", "Stopped"}
deadline = time.time() + 2 * 60 * 60
last = ""
while time.time() < deadline:
    desc = sm.describe_ai_recommendation_job(AIRecommendationJobName=REC_JOB)
    status = desc["AIRecommendationJobStatus"]
    if status != last:
        print(f"  [{time.strftime('%H:%M:%S')}] {status}")
        last = status
    if status in TERMINAL:
        break
    time.sleep(60)
if status == "Failed":
    print(f"  FailureReason: {desc.get('FailureReason')}")
elif status == "Completed":
    for i, rec in enumerate(desc.get("Recommendations", [])):
        deploy = rec.get("DeploymentConfiguration", {})
        opts = [d.get("OptimizationType") for d in rec.get("OptimizationDetails", [])]
        print(f"  [{i}] {deploy.get('InstanceType')} — {opts}")

## Step 8: Verify in MLflow

Open the MLflow App UI, navigate to your experiment, and compare the benchmark and recommendation runs.

In [ ]:
print(f"Open the MLflow App in the AWS Console and navigate to experiment: {MLFLOW_EXPERIMENT}")
print(f"You'll see runs: benchmark-{{uid}} and recommendation-{{uid}}")

## Step 9: Cleanup

Delete the jobs and workload configs created during this tutorial.

In [ ]:
# --- Cleanup ---
def _try(label, fn):
    try:
        fn(); print(f"  deleted {label}")
    except Exception as e:
        print(f"  skip {label}: {type(e).__name__}: {e}")

_try("benchmark job",     lambda: sm.delete_ai_benchmark_job(AIBenchmarkJobName=BENCH_JOB))
_try("benchmark config",  lambda: sm.delete_ai_workload_config(AIWorkloadConfigName=BENCH_CONFIG))
_try("recommendation",    lambda: sm.delete_ai_recommendation_job(AIRecommendationJobName=REC_JOB))
_try("rec config",        lambda: sm.delete_ai_workload_config(AIWorkloadConfigName=REC_CONFIG))

## Reference

### MlflowConfig Fields

| Field | Notes |
|-------|-------|
| `MlflowResourceArn` | ARN of the MLflow App (required) |
| `MlflowExperimentName` | Auto-created if it doesn't exist |
| `MlflowRunName` | Defaults to the job name if omitted |

### IAM Role Requirements

The execution role's **trust policy** must allow both principals:
- `arn:aws:iam::<account>:root` — so the service's internal Lambda can assume it
- `sagemaker.amazonaws.com` — so SageMaker can assume it during training/endpoint creation

**Managed policy:** `AmazonSageMakerFullAccess`

**Inline policies:**
- `MlflowDataPlaneAccess` — `sagemaker-mlflow:*`, `sagemaker:CallMlflowAppApi`, `sagemaker:DescribeMlflowApp`
- `ServiceQuotasAccess` — `servicequotas:GetServiceQuota` (probed at job-submit time)
- `EndpointInvokeAccess` — `sagemaker:InvokeEndpoint*` on your endpoint ARN

### Tips

- Set `tooling.version` ≥ `0.8.0` for MLflow nested-run support
- These APIs integrate with **SageMaker MLflow App** only — not self-hosted MLflow servers
- Recommendation jobs provision their own endpoints; omit `ComputeSpec.InstanceTypes` unless you have a specific instance in mind